# 🫀 Multimodal Heart Failure Readmission Prediction Pipeline
**Master Google Colab Notebook with Google Drive Backup**

This notebook mounts Google Drive for persistent output backup and executes the full pipeline:

1. **Mount Google Drive** & Create Backup Directory
2. **Install Required Python Packages** (`timm`, `wfdb`, `xgboost`, `torchxrayvision`, `shap`, `dcurves`)
3. **Setup & Clone Repository**
4. **Copy Data / PhysioNet Download**
5. **Train Tabular Branch** (XGBoost Ensemble with Trajectories & Ratios)
6. **Train ECG Branch** (1D ResNet-34 with Lead Attention)
7. **Train CXR Branch** (TorchXRayVision DenseNet-121 Medical Pretraining)
8. **Train Gated Fusion Layer** (Focal Loss & Gated MLP)
9. **Run Comprehensive Evaluation Suite** (DCA, Baselines, Confusion Matrices, Fairness)
10. **Auto-Backup Outputs to Google Drive** & Render Dashboard Inline

In [ ]:
# ── Cell 1: Mount Google Drive & Setup Backup Folder ───────────────────
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/HealthCare_Analytics_Backup'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f'Google Drive mounted successfully! Backup folder: {DRIVE_DIR}')
except Exception as e:
    print(f'Running in non-Colab environment: {e}')

In [ ]:
# ── Cell 2: Install Required Python Packages ────────────────────────────
!pip install -q timm wfdb xgboost shap dcurves pyarrow torchxrayvision

In [ ]:
# ── Cell 3: Setup & Clone Repository ──────────────────────────────────
import os, shutil
if not os.path.exists('HealthCare_Analytics'):
    !git clone https://github.com/KMohnishM/HealthCare_Analytics.git
%cd HealthCare_Analytics
!git pull

In [ ]:
# ── Cell 4: Copy Existing Data from Drive or Download from PhysioNet ───
import os
os.makedirs('data', exist_ok=True)
DRIVE_DIR = '/content/drive/MyDrive/HealthCare_Analytics_Backup'

if os.path.exists(f'{DRIVE_DIR}/data'):
    print('Copying existing dataset from Google Drive...')
    !cp -r {DRIVE_DIR}/data/* data/ 2>/dev/null || true

print('\n--- Current Data Directory Contents ---')
!ls -la data

print('\nChecking / downloading PhysioNet cohort files...')
!python scripts/download_cohort_physionet.py --cohort data/cohort.parquet --username kmohnishm --password HereisMy2006Bye

In [ ]:
# ── Cell 5: Train Tabular Branch ───────────────────────────────────────
!python scripts/train_tabular.py

In [ ]:
# ── Cell 6: Train ECG Branch ───────────────────────────────────────────
!python scripts/train_ecg.py

In [ ]:
# ── Cell 7: Train CXR Branch ───────────────────────────────────────────
!python scripts/train_cxr.py

In [ ]:
# ── Cell 8: Train Gated Fusion Layer ───────────────────────────────────
!python scripts/train_fusion.py

In [ ]:
# ── Cell 9: Run Evaluation Suite & DCA ─────────────────────────────────
!python scripts/evaluate_all.py

In [ ]:
# ── Cell 10: Auto-Backup Outputs to Google Drive ────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/HealthCare_Analytics_Backup'
if os.path.exists('/content/drive/MyDrive'):
    print(f'Backing up model weights and evaluation figures to {DRIVE_DIR} ...')
    os.makedirs(f'{DRIVE_DIR}/outputs', exist_ok=True)
    os.makedirs(f'{DRIVE_DIR}/data', exist_ok=True)
    !cp -r outputs/* {DRIVE_DIR}/outputs/
    !cp -r data/* {DRIVE_DIR}/data/ 2>/dev/null || true
    print('Backup to Google Drive complete!')
else:
    print('Google Drive not mounted — skipping backup step.')

In [ ]:
# ── Cell 11: Display Inline Evaluation Dashboard ───────────────────────
import os
from IPython.display import Image, display, HTML

figures = [
    ('confusion_matrix.png', 'Side-by-Side Confusion Matrices (F1-Optimized Thresholds)'),
    ('decision_curve.png', 'Clinical Decision Curve Analysis (DCA vs. LACE / HOSPITAL)'),
    ('missingness_sweep_heatmap.png', 'Modality Missingness Sweep Heatmap'),
    ('fairness_subgroups.png', 'Algorithmic Fairness Subgroup Analysis')
]

for filename, title in figures:
    filepath = os.path.join('outputs', 'figures', filename)
    if os.path.exists(filepath):
        display(HTML(f"<h3 style='color:#2c3e50; font-family:sans-serif;'>{title} (<code>{filename}</code>)</h3>"))
        display(Image(filename=filepath, width=750))
    else:
        print(f'Warning: Figure {filename} not found at {filepath}')